---
## Notebook para tratamento das variáveis explicativas

- Tratamento das variáveis respostas: '/workspaces/CEA-1/análise de dados/tratamento_da_variavel_resposta.ipynb'

- Visualização inicial das variáveis: '/workspaces/CEA-1/análise de dados/cea_analise_inicial_v0.ipynb'

- Relação das variáveis iniciais: '/workspaces/CEA-1/drafts/tratamento_de_dados.ipynb']


**Novos resultados**

- regiao: substituir 'dt_not'

---
### Imports


In [1]:
import re
import pandas as pd
import matplotlib.pyplot as plt


In [2]:
%%capture
%run '/workspaces/CEA-1/análise de dados/tratamento_da_variavel_resposta.ipynb'

---
### Tratamentos

**1. dt_not**

- Tranformação para **dt_not_ano**, que representa o ano da data de notificação

In [90]:
dados['dt_not'] = pd.to_datetime(dados['dt_not'])
dados['dt_not_ano']  = dados['dt_not'].dt.year
dados['dt_not_ano'].value_counts().sort_values(ascending=False)


dt_not_ano
2018    336
2019    301
2020    273
2021    125
2022     47
2023      1
Name: count, dtype: int64

In [91]:
dados = dados.drop(columns=['dt_not'])

**2. preench**
- Transformação para **preench_ano**, que representa o ano de preenchimento da notificação

In [92]:
dados['preench'] = pd.to_datetime(dados['preench'])
dados['preench_ano']  = dados['preench'].dt.year
dados['preench_ano'].value_counts().sort_values(ascending=False)

/tmp/ipykernel_36993/878385915.py:1: UserWarning: Parsing dates in %d/%m/%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  dados['preench'] = pd.to_datetime(dados['preench'])


preench_ano
2018    336
2019    301
2020    273
2021    125
2022     47
2023      1
Name: count, dtype: int64

In [93]:
dados = dados.drop(columns=['preench'])

**3. uf_not**
- Transformação para **regiao**, que representa a região do estado

In [94]:
mapa_regiao = {
    'AC': 'Norte', 'AM': 'Norte', 'AP': 'Norte', 'PA': 'Norte', 'RO': 'Norte', 'RR': 'Norte', 'TO': 'Norte',
    'AL': 'Nordeste', 'BA': 'Nordeste', 'CE': 'Nordeste', 'MA': 'Nordeste', 'PB': 'Nordeste', 'PE': 'Nordeste', 'RN': 'Nordeste', 'SE': 'Nordeste',
    'DF': 'Centro-Oeste', 'GO': 'Centro-Oeste', 'MS': 'Centro-Oeste', 'MT': 'Centro-Oeste',
    'ES': 'Sudeste', 'MG': 'Sudeste', 'RJ': 'Sudeste', 'SP': 'Sudeste',
    'PR': 'Sul', 'RS': 'Sul', 'SC': 'Sul'
}

# Criar coluna 'regiao' baseada em 'uf_not'
dados['regiao'] = dados['uf_not'].map(mapa_regiao)

print(dados['regiao'].value_counts().sort_values(ascending=False))


regiao
Sudeste         592
Sul             308
Nordeste         97
Centro-Oeste     46
Norte            40
Name: count, dtype: int64


In [95]:
dados = dados.drop(columns=['uf_not'])

**4. pais_nasc**


In [96]:
dados['pais_nasc'] = dados['pais_nasc'].replace(' ', 'nao informado')
print(dados['pais_nasc'].value_counts().sort_values(ascending=False))

pais_nasc
BRASIL           929
nao informado    150
VENEZUELA          2
OUTROS PAISES      1
PERU               1
Name: count, dtype: int64


**5. sexo**

In [97]:
# Substituindo os valores 1 e 2 pelos textos correspondentes
dados['sexo'] = dados['sexo'].replace({1: 'feminino', 2: 'masculino'})

# Imprimindo a contagem novamente para conferir o resultado
print(dados['sexo'].value_counts().sort_values(ascending=False))

sexo
masculino    667
feminino     416
Name: count, dtype: int64


**6. cor**

In [98]:
dados['cor'] = dados['cor'].replace(' ', 'Nao Informado')
# Criando o dicionário com o de-para das cores
mapeamento_cor = {
    1: 'Branca', '1': 'Branca',
    2: 'Preta',  '2': 'Preta',
    3: 'Parda',  '3': 'Parda',
    4: 'Amarela', '4': 'Amarela',
    5: 'Indígena', '5': 'Indígena',
    9: 'Nao Informado', '9': 'Nao Informado'
}

# Aplicando a substituição
dados['cor'] = dados['cor'].replace(mapeamento_cor)

# Imprimindo a contagem para verificar o resultado
print(dados['cor'].value_counts().sort_values(ascending=False))

cor
Nao Informado    725
Branca           188
Parda             99
Amarela           61
Negra              8
Indígena           2
Name: count, dtype: int64


**7. gestante**

In [99]:
dados['gestante'] = dados['gestante'].replace({1: 'Sim', 2: 'Nao', 9: 'Nao Informado'})
print(dados['gestante'].value_counts().sort_values(ascending=False))

gestante
Nao Informado    969
Nao              109
Sim                5
Name: count, dtype: int64


**8. cod_prod**

Transformação em **cod_prod_ajustada**, que é um valor binário de "tomou vacina 67 isoladamente ou tomou outras também"

In [100]:
dados['cod_prod_ajustada'] = dados['cod_prod'].apply(
    lambda x: 'apenas 67' if str(x).strip() == '67' else 'outros com 67'
)

print(dados['cod_prod_ajustada'].value_counts())

cod_prod_ajustada
apenas 67        720
outros com 67    363
Name: count, dtype: int64


In [101]:
dados = dados.drop(columns=['cod_prod'])

**9. dt_apl**

Transformação em **dt_apl_ano**, que é o ano da aplicação

In [102]:
dados['dt_apl'] = dados['dt_apl'].str.extract(r'(\d{2}/\d{2}/\d{4})')
dados['dt_apl'] = pd.to_datetime(dados['dt_apl'], format='%d/%m/%Y')

dados['dt_apl_ano']    = dados['dt_apl'].dt.year

print(dados['dt_apl_ano'].value_counts())

dt_apl_ano
2018    328
2019    296
2020    225
2021    116
2017     72
2022     31
2015      6
2016      5
2014      2
2023      1
2006      1
Name: count, dtype: int64


In [103]:
dados = dados.drop(columns=['dt_apl'])

**10. lote**

In [104]:
print(dados['lote'].value_counts().sort_values(ascending=False))

lote
(67) 170013                                                                                                     31
(67) 180010                                                                                                     28
(67) 180018                                                                                                     24
(67) 180017                                                                                                     21
(67) 190007                                                                                                     20
(67) 190012                                                                                                     17
(67) 180233                                                                                                     17
(67) 180231                                                                                                     17
(67) 170137                                                                

In [105]:
import re

def extrair_lote_67(valor):
    if pd.isna(valor):
        return None
    matches = re.findall(r'\(67\)\s*(\S+)', str(valor))
    return ' | '.join(matches) if matches else None
    
dados['lote_ajustada'] = dados['lote'].apply(extrair_lote_67)
print(dados['lote_ajustada'].value_counts().sort_values(ascending=False))

lote_ajustada
170013               47
180010               43
180018               33
180017               32
190012               30
180048               29
180231               27
170137               26
190005               25
190007               25
160137               24
190011               22
180200               22
180044               21
170014               21
170136               21
180202               20
180233               20
180046               20
170016               17
170044               17
160140               17
190105               16
180198               16
190101               14
180032               14
180034               14
190102               13
190013               13
180199               13
190104               12
190090               12
180042               12
200249               11
180201               10
Não                  10
180232                9
170017                9
190202                8
190154                8
190014                8
17

In [106]:
dados = dados.drop(columns=['lote'])

In [ ]:
dados = dados.drop(columns=['lote_ajustada'])

**11. dose**

Transformação em **dose_ajustada**, que é junção das infos

In [107]:
dados['dose_ajustada'] = dados['dose'].apply(
    lambda x: 'apenas 67' if str(x).strip() == '(67) 1ª Dose' else 'outros com 67'
)

print(dados['dose_ajustada'].value_counts())

dose_ajustada
apenas 67        720
outros com 67    363
Name: count, dtype: int64


In [108]:
dados = dados.drop(columns=['dose'])

**12. via_adm**

Transformação em **via_adm**, que é junção das infos

In [109]:
import numpy as np

via_67 = dados['via_adm'].astype(str).str.extract(r'\(67\)\s*([^|]*)')[0].str.strip()

dados['via_adm_ajustada'] = np.select(
    [via_67 == 'Intramuscular', via_67 == ''],
    ['intramuscular', 'vazio'],
    default='outros'
)

print(dados['via_adm_ajustada'].value_counts())

via_adm_ajustada
intramuscular    882
vazio            194
outros             7
Name: count, dtype: int64


In [110]:
dados = dados.drop(columns=['via_adm'])

**13. local_aplic**

Transformação em **local_aplic_ajustada**, que é junção das infos

In [111]:
local_67 = dados['local_aplic'].astype(str).str.extract(r'\(67\)\s*([^|]*)')[0].str.strip()

dados['local_aplic_ajustada'] = local_67.replace('', 'não informado')

print(dados['local_aplic_ajustada'].value_counts())

local_aplic_ajustada
Deltóide Esquerdo                 466
Deltóide Direito                  348
não informado                     194
Ventroglúteo Direito               48
Ventroglúteo Esquerdo              19
Vasto Lateral da Coxa Esquerda      5
Vasto Lateral da Coxa Direito       2
Glúteo                              1
Name: count, dtype: int64


In [112]:
dados = dados.drop(columns=['local_aplic'])

**14.tp_med**

In [113]:
# Se tiver certeza de que só existem os valores 1 e 2 (e quiser usar map)
dados['tp_med'] = dados['tp_med'].map({1: 'Sim', 2: 'Nao', 9: 'Ignorado'})

In [114]:
print(dados['tp_med'].value_counts())

tp_med
Ignorado    628
Sim         310
Nao         145
Name: count, dtype: int64


**15. esavi_noti**

In [115]:
import re
import pandas as pd
import unicodedata

def normalizar(texto):
    texto = texto.lower().strip()
    texto = unicodedata.normalize('NFKD', texto)
    return ''.join(c for c in texto if not unicodedata.combining(c))

def make_pattern(keywords):
    return '|'.join(re.escape(k) for k in keywords)

# ── keywords (sem 'dor' genérico na lista) ──────────────────────────────────

dor_keywords          = ['dor no local da aplicacao', 'dor no deltoide', 'dor no braco',
                          'dor em membro', 'dor em mmii', 'dor na panturrilha', 'dor cervical']
dor_abdominal_kw      = ['dor abdominal', 'dor epigastrica', 'dor uterina']
dor_corpo_kw          = ['dor no corpo', 'dor muscular', 'mialgia']
artralgia_keywords    = ['poliartralgia', 'dor nas articulacoes', 'artralgia']
cefaleia_keywords     = ['dor de cabeca', 'cefaleia']
edema_keywords        = ['edemaciado', 'intumescimento', 'inchaco', 'edema']
eritema_keywords      = ['manchas vermelhas', 'vermelhidao', 'hiperemia', 'rubor', 'eritema']
calor_keywords        = ['hiperemia e calor', 'calor']
enduracao_keywords    = ['endurecimento', 'enduracao']
abscesso_keywords     = ['abscesso quente', 'exsudado purulento', 'exsudado seroso', 'celulite']
lesao_keywords        = ['lesoes cutaneas', 'lesao no braco', 'hematoma', 'cisto',
                          'bolhas', 'pustulas', 'lesao', 'lesoes']
linfono_keywords      = ['linfono em tamanho aumentado', 'linfonodo aumentado', 'linfonodomegalia']
prurido_keywords      = ['prurido']
exantema_local_kw     = ['exantema no local da aplicacao', 'exantema em deltoide']
exantema_keywords     = ['exantema generalizado', 'exantema']
febre_keywords        = ['hipertermia', 'sensacao febril', 'febre']
nausea_keywords       = ['ansia de vomito', 'nausea', 'enjoo']
emese_keywords        = ['emese', 'vomito']
diarreia_keywords     = ['quadro diarreico', 'diarreia']
tontura_keywords      = ['lightheaded', 'lipotimia', 'vertigem', 'tontura']
sincope_keywords      = ['sincope vasovagal', 'desmaio', 'sincope']
parestesia_keywords   = ['amortecimento', 'dormencia', 'parestesia']
convulsao_keywords    = ['crise convulsiva', 'crise de ausencia', 'convulsao']
confusao_keywords     = ['hiporresponsividade transitoria', 'confusao mental']
fraqueza_keywords     = ['dificuldade de deambular', 'astenia', 'letargia', 'moleza', 'fraqueza']
hipotensao_keywords   = ['queda de pa', 'pa baixa', 'hipotensao']
taquicardia_keywords  = ['taquicardia']
bradicardia_keywords  = ['bradicardia']
extremidades_keywords = ['extremidades frias', 'cianose']
palidez_keywords      = ['hipocorada', 'descorada', 'palidez']
sudorese_keywords     = ['suor frio', 'sudorese']
urticaria_keywords    = ['maculas eritematosas pruriginosas', 'papulas', 'bolinhas',
                          'coceira', 'urticaria']
broncoespasmo_kw      = ['broncoespasmo']
dispneia_keywords     = ['dificuldade para respirar', 'falta de ar', 'dispneia']
angioedema_keywords   = ['angioedema']
tremor_keywords       = ['tremores', 'tremor']
fotofobia_keywords    = ['sensibilidade a luz', 'fotofobia']
visao_turva_keywords  = ['visao embacada', 'vista escura', 'visao turva']
guillain_barre_kw     = ['sindrome de guillain-barre', 'guillain-barre', 'guillain barre']
encefalite_keywords   = ['encefalite']
epilepsia_keywords    = ['epilepsia']
paralisia_keywords    = ['paralisia']
purpura_keywords      = ['purpura trombocitopenica', 'purpura']

# ── pattern especial para 'dor' genérico ────────────────────────────────────
# \bdor\b só bate se NÃO vier seguido de qualificador de subcategoria

_dor_exclusoes = (
    r'abdominal|epigastrica|uterina|'   # dor abdominal
    r'muscular|no corpo|'               # dor no corpo
    r'nas articulacoes|'                # artralgia
    r'de cabeca'                        # cefaleia
)

dor_generico_pattern = (
    make_pattern(dor_keywords) +                            # keywords específicas de 'Dor'
    r'|\bdor\b(?!\s*(?:' + _dor_exclusoes + r'))'          # 'dor' sozinho sem qualificador de subcategoria
)

# ── colunas_categorias ───────────────────────────────────────────────────────

colunas_categorias = [
    ('esavi_dor_abdominal',             make_pattern(dor_abdominal_kw)),
    ('esavi_dor_no_corpo',              make_pattern(dor_corpo_kw)),
    ('esavi_artralgia',                 make_pattern(artralgia_keywords)),
    ('esavi_cefaleia',                  make_pattern(cefaleia_keywords)),
    ('esavi_dor',                       dor_generico_pattern),              # ← pattern especial
    ('esavi_exantema_local',            make_pattern(exantema_local_kw)),
    ('esavi_exantema',                  make_pattern(exantema_keywords)),
    ('esavi_edema',                     make_pattern(edema_keywords)),
    ('esavi_eritema',                   make_pattern(eritema_keywords)),
    ('esavi_calor',                     make_pattern(calor_keywords)),
    ('esavi_enduracao',                 make_pattern(enduracao_keywords)),
    ('esavi_abscesso_quente',           make_pattern(abscesso_keywords)),
    ('esavi_lesao',                     make_pattern(lesao_keywords)),
    ('esavi_linfonodomegalia',          make_pattern(linfono_keywords)),
    ('esavi_prurido',                   make_pattern(prurido_keywords)),
    ('esavi_febre',                     make_pattern(febre_keywords)),
    ('esavi_nausea',                    make_pattern(nausea_keywords)),
    ('esavi_emese',                     make_pattern(emese_keywords)),
    ('esavi_diarreia',                  make_pattern(diarreia_keywords)),
    ('esavi_tontura',                   make_pattern(tontura_keywords)),
    ('esavi_sincope',                   make_pattern(sincope_keywords)),
    ('esavi_parestesia',                make_pattern(parestesia_keywords)),
    ('esavi_convulsao',                 make_pattern(convulsao_keywords)),
    ('esavi_confusao_mental',           make_pattern(confusao_keywords)),
    ('esavi_fraqueza',                  make_pattern(fraqueza_keywords)),
    ('esavi_hipotensao',                make_pattern(hipotensao_keywords)),
    ('esavi_taquicardia',               make_pattern(taquicardia_keywords)),
    ('esavi_bradicardia',               make_pattern(bradicardia_keywords)),
    ('esavi_extremidades_frias',        make_pattern(extremidades_keywords)),
    ('esavi_palidez',                   make_pattern(palidez_keywords)),
    ('esavi_sudorese',                  make_pattern(sudorese_keywords)),
    ('esavi_urticaria',                 make_pattern(urticaria_keywords)),
    ('esavi_broncoespasmo',             make_pattern(broncoespasmo_kw)),
    ('esavi_dispneia',                  make_pattern(dispneia_keywords)),
    ('esavi_angioedema',                make_pattern(angioedema_keywords)),
    ('esavi_tremor',                    make_pattern(tremor_keywords)),
    ('esavi_fotofobia',                 make_pattern(fotofobia_keywords)),
    ('esavi_visao_turva',               make_pattern(visao_turva_keywords)),
    ('esavi_guillain_barre',            make_pattern(guillain_barre_kw)),
    ('esavi_encefalite',                make_pattern(encefalite_keywords)),
    ('esavi_epilepsia',                 make_pattern(epilepsia_keywords)),
    ('esavi_paralisia',                 make_pattern(paralisia_keywords)),
    ('esavi_purpura_trombocitopenica',  make_pattern(purpura_keywords)),
]

# ── criação das variáveis binárias ───────────────────────────────────────────

for coluna, pattern in colunas_categorias:
    dados[coluna] = dados['esavi_noti'].apply(
        lambda x, p=pattern: 1 if pd.notna(x) and
        re.search(p, normalizar(str(x)), re.IGNORECASE) else 0
    )

In [116]:
# ── checagem rápida ─────────────────────────────────────────────────────────
print(dados[[col for col, _ in colunas_categorias]].sum().sort_values(ascending=False))

esavi_dor                         122
esavi_eritema                     114
esavi_edema                       108
esavi_febre                       104
esavi_emese                        95
esavi_cefaleia                     94
esavi_sincope                      79
esavi_prurido                      61
esavi_nausea                       58
esavi_palidez                      56
esavi_calor                        56
esavi_tontura                      48
esavi_urticaria                    37
esavi_exantema                     31
esavi_sudorese                     29
esavi_fraqueza                     27
esavi_dor_abdominal                26
esavi_convulsao                    25
esavi_hipotensao                   24
esavi_diarreia                     23
esavi_lesao                        22
esavi_dor_no_corpo                 15
esavi_parestesia                   14
esavi_dispneia                     12
esavi_extremidades_frias           12
esavi_visao_turva                  11
esavi_tremor

**15. cls_ei_ajustada**

In [117]:
import re

def ajustar_cls_ei(valor):
    if pd.isna(valor) or str(valor).strip() == "":
        return "Não se aplica"
    match = re.match(r"(A\.3\.[0-9]+)", str(valor))
    if match:
        return match.group(1)
    return str(valor)

dados["cls_ei_ajustada"] = dados["cls_ei"].apply(ajustar_cls_ei)

print(dados["cls_ei_ajustada"].value_counts())

cls_ei_ajustada
Não se aplica    583
A.3.7            306
A.3.4             71
A.3.9             49
A.3.5             42
A.3.2             11
A.3.6              8
A.3.8              7
A.3.1              5
A.3.3              1
Name: count, dtype: int64


In [118]:
dados = dados.drop(columns=['cls_ei'])

**16. cls_erro (retirado)**

In [69]:
dados = dados.drop(columns=['cls_erro'])

**17. evol_caso**

In [70]:
print(dados['evol_caso'].value_counts())

evol_caso
                       588
Cura sem sequelas      372
Em acompanhamento      100
Cura com sequelas       11
Não é EAPV               7
Perda de seguimento      5
Name: count, dtype: int64


In [71]:
dados = dados.drop(columns=['evol_caso'])

**18. man_loc**

In [72]:
colunas_categorias_man_loc = [
    ('man_loc_dor_abdominal',             make_pattern(dor_abdominal_kw)),
    ('man_loc_dor_no_corpo',              make_pattern(dor_corpo_kw)),
    ('man_loc_artralgia',                 make_pattern(artralgia_keywords)),
    ('man_loc_cefaleia',                  make_pattern(cefaleia_keywords)),
    ('man_loc_dor',                       dor_generico_pattern),
    ('man_loc_exantema_local',            make_pattern(exantema_local_kw)),
    ('man_loc_exantema',                  make_pattern(exantema_keywords)),
    ('man_loc_edema',                     make_pattern(edema_keywords)),
    ('man_loc_eritema',                   make_pattern(eritema_keywords)),
    ('man_loc_calor',                     make_pattern(calor_keywords)),
    ('man_loc_enduracao',                 make_pattern(enduracao_keywords)),
    ('man_loc_abscesso_quente',           make_pattern(abscesso_keywords)),
    ('man_loc_lesao',                     make_pattern(lesao_keywords)),
    ('man_loc_linfonodomegalia',          make_pattern(linfono_keywords)),
    ('man_loc_prurido',                   make_pattern(prurido_keywords)),
    ('man_loc_febre',                     make_pattern(febre_keywords)),
    ('man_loc_nausea',                    make_pattern(nausea_keywords)),
    ('man_loc_emese',                     make_pattern(emese_keywords)),
    ('man_loc_diarreia',                  make_pattern(diarreia_keywords)),
    ('man_loc_tontura',                   make_pattern(tontura_keywords)),
    ('man_loc_sincope',                   make_pattern(sincope_keywords)),
    ('man_loc_parestesia',                make_pattern(parestesia_keywords)),
    ('man_loc_convulsao',                 make_pattern(convulsao_keywords)),
    ('man_loc_confusao_mental',           make_pattern(confusao_keywords)),
    ('man_loc_fraqueza',                  make_pattern(fraqueza_keywords)),
    ('man_loc_hipotensao',                make_pattern(hipotensao_keywords)),
    ('man_loc_taquicardia',               make_pattern(taquicardia_keywords)),
    ('man_loc_bradicardia',               make_pattern(bradicardia_keywords)),
    ('man_loc_extremidades_frias',        make_pattern(extremidades_keywords)),
    ('man_loc_palidez',                   make_pattern(palidez_keywords)),
    ('man_loc_sudorese',                  make_pattern(sudorese_keywords)),
    ('man_loc_urticaria',                 make_pattern(urticaria_keywords)),
    ('man_loc_broncoespasmo',             make_pattern(broncoespasmo_kw)),
    ('man_loc_dispneia',                  make_pattern(dispneia_keywords)),
    ('man_loc_angioedema',                make_pattern(angioedema_keywords)),
    ('man_loc_tremor',                    make_pattern(tremor_keywords)),
    ('man_loc_fotofobia',                 make_pattern(fotofobia_keywords)),
    ('man_loc_visao_turva',               make_pattern(visao_turva_keywords)),
    ('man_loc_guillain_barre',            make_pattern(guillain_barre_kw)),
    ('man_loc_encefalite',                make_pattern(encefalite_keywords)),
    ('man_loc_epilepsia',                 make_pattern(epilepsia_keywords)),
    ('man_loc_paralisia',                 make_pattern(paralisia_keywords)),
    ('man_loc_purpura_trombocitopenica',  make_pattern(purpura_keywords)),
]

for coluna, pattern in colunas_categorias_man_loc:
    dados[coluna] = dados['man_loc'].apply(
        lambda x, p=pattern: 1 if pd.notna(x) and
        re.search(p, normalizar(str(x)), re.IGNORECASE) else 0
    )

# ── checagem rápida ──────────────────────────────────────────────────────────
print(dados[[col for col, _ in colunas_categorias_man_loc]].sum().sort_values(ascending=False))

man_loc_dor                         144
man_loc_eritema                     107
man_loc_edema                        97
man_loc_calor                        83
man_loc_abscesso_quente              33
man_loc_prurido                      31
man_loc_urticaria                    15
man_loc_lesao                         5
man_loc_febre                         3
man_loc_enduracao                     2
man_loc_fraqueza                      1
man_loc_paralisia                     1
man_loc_hipotensao                    1
man_loc_sincope                       1
man_loc_cefaleia                      0
man_loc_dor_no_corpo                  0
man_loc_dor_abdominal                 0
man_loc_exantema                      0
man_loc_artralgia                     0
man_loc_exantema_local                0
man_loc_emese                         0
man_loc_diarreia                      0
man_loc_tontura                       0
man_loc_convulsao                     0
man_loc_parestesia                    0


**19. man_sis**

In [73]:
colunas_categorias_man_sis = [
    ('man_sis_dor_abdominal',             make_pattern(dor_abdominal_kw)),
    ('man_sis_dor_no_corpo',              make_pattern(dor_corpo_kw)),
    ('man_sis_artralgia',                 make_pattern(artralgia_keywords)),
    ('man_sis_cefaleia',                  make_pattern(cefaleia_keywords)),
    ('man_sis_dor',                       dor_generico_pattern),
    ('man_sis_exantema_local',            make_pattern(exantema_local_kw)),
    ('man_sis_exantema',                  make_pattern(exantema_keywords)),
    ('man_sis_edema',                     make_pattern(edema_keywords)),
    ('man_sis_eritema',                   make_pattern(eritema_keywords)),
    ('man_sis_calor',                     make_pattern(calor_keywords)),
    ('man_sis_enduracao',                 make_pattern(enduracao_keywords)),
    ('man_sis_abscesso_quente',           make_pattern(abscesso_keywords)),
    ('man_sis_lesao',                     make_pattern(lesao_keywords)),
    ('man_sis_linfonodomegalia',          make_pattern(linfono_keywords)),
    ('man_sis_prurido',                   make_pattern(prurido_keywords)),
    ('man_sis_febre',                     make_pattern(febre_keywords)),
    ('man_sis_nausea',                    make_pattern(nausea_keywords)),
    ('man_sis_emese',                     make_pattern(emese_keywords)),
    ('man_sis_diarreia',                  make_pattern(diarreia_keywords)),
    ('man_sis_tontura',                   make_pattern(tontura_keywords)),
    ('man_sis_sincope',                   make_pattern(sincope_keywords)),
    ('man_sis_parestesia',                make_pattern(parestesia_keywords)),
    ('man_sis_convulsao',                 make_pattern(convulsao_keywords)),
    ('man_sis_confusao_mental',           make_pattern(confusao_keywords)),
    ('man_sis_fraqueza',                  make_pattern(fraqueza_keywords)),
    ('man_sis_hipotensao',                make_pattern(hipotensao_keywords)),
    ('man_sis_taquicardia',               make_pattern(taquicardia_keywords)),
    ('man_sis_bradicardia',               make_pattern(bradicardia_keywords)),
    ('man_sis_extremidades_frias',        make_pattern(extremidades_keywords)),
    ('man_sis_palidez',                   make_pattern(palidez_keywords)),
    ('man_sis_sudorese',                  make_pattern(sudorese_keywords)),
    ('man_sis_urticaria',                 make_pattern(urticaria_keywords)),
    ('man_sis_broncoespasmo',             make_pattern(broncoespasmo_kw)),
    ('man_sis_dispneia',                  make_pattern(dispneia_keywords)),
    ('man_sis_angioedema',                make_pattern(angioedema_keywords)),
    ('man_sis_tremor',                    make_pattern(tremor_keywords)),
    ('man_sis_fotofobia',                 make_pattern(fotofobia_keywords)),
    ('man_sis_visao_turva',               make_pattern(visao_turva_keywords)),
    ('man_sis_guillain_barre',            make_pattern(guillain_barre_kw)),
    ('man_sis_encefalite',                make_pattern(encefalite_keywords)),
    ('man_sis_epilepsia',                 make_pattern(epilepsia_keywords)),
    ('man_sis_paralisia',                 make_pattern(paralisia_keywords)),
    ('man_sis_purpura_trombocitopenica',  make_pattern(purpura_keywords)),
]

for coluna, pattern in colunas_categorias_man_sis:
    dados[coluna] = dados['man_sis'].apply(
        lambda x, p=pattern: 1 if pd.notna(x) and
        re.search(p, normalizar(str(x)), re.IGNORECASE) else 0
    )

# ── checagem rápida ──────────────────────────────────────────────────────────
print(dados[[col for col, _ in colunas_categorias_man_sis]].sum().sort_values(ascending=False))

man_sis_palidez                     82
man_sis_sincope                     66
man_sis_exantema                    55
man_sis_hipotensao                  38
man_sis_prurido                     34
man_sis_convulsao                   24
man_sis_fraqueza                    21
man_sis_urticaria                   21
man_sis_extremidades_frias          17
man_sis_eritema                     13
man_sis_edema                       12
man_sis_dispneia                    10
man_sis_angioedema                  10
man_sis_paralisia                   10
man_sis_taquicardia                  8
man_sis_parestesia                   8
man_sis_sudorese                     4
man_sis_bradicardia                  4
man_sis_dor                          4
man_sis_tontura                      3
man_sis_tremor                       2
man_sis_cefaleia                     2
man_sis_lesao                        2
man_sis_purpura_trombocitopenica     1
man_sis_febre                        1
man_sis_visao_turva      

/tmp/ipykernel_36993/3020130088.py:48: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  dados[coluna] = dados['man_sis'].apply(
/tmp/ipykernel_36993/3020130088.py:48: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  dados[coluna] = dados['man_sis'].apply(
/tmp/ipykernel_36993/3020130088.py:48: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame

**20. man_out**

In [74]:
colunas_categorias_man_out = [
    ('man_out_dor_abdominal',             make_pattern(dor_abdominal_kw)),
    ('man_out_dor_no_corpo',              make_pattern(dor_corpo_kw)),
    ('man_out_artralgia',                 make_pattern(artralgia_keywords)),
    ('man_out_cefaleia',                  make_pattern(cefaleia_keywords)),
    ('man_out_dor',                       dor_generico_pattern),
    ('man_out_exantema_local',            make_pattern(exantema_local_kw)),
    ('man_out_exantema',                  make_pattern(exantema_keywords)),
    ('man_out_edema',                     make_pattern(edema_keywords)),
    ('man_out_eritema',                   make_pattern(eritema_keywords)),
    ('man_out_calor',                     make_pattern(calor_keywords)),
    ('man_out_enduracao',                 make_pattern(enduracao_keywords)),
    ('man_out_abscesso_quente',           make_pattern(abscesso_keywords)),
    ('man_out_lesao',                     make_pattern(lesao_keywords)),
    ('man_out_linfonodomegalia',          make_pattern(linfono_keywords)),
    ('man_out_prurido',                   make_pattern(prurido_keywords)),
    ('man_out_febre',                     make_pattern(febre_keywords)),
    ('man_out_nausea',                    make_pattern(nausea_keywords)),
    ('man_out_emese',                     make_pattern(emese_keywords)),
    ('man_out_diarreia',                  make_pattern(diarreia_keywords)),
    ('man_out_tontura',                   make_pattern(tontura_keywords)),
    ('man_out_sincope',                   make_pattern(sincope_keywords)),
    ('man_out_parestesia',                make_pattern(parestesia_keywords)),
    ('man_out_convulsao',                 make_pattern(convulsao_keywords)),
    ('man_out_confusao_mental',           make_pattern(confusao_keywords)),
    ('man_out_fraqueza',                  make_pattern(fraqueza_keywords)),
    ('man_out_hipotensao',                make_pattern(hipotensao_keywords)),
    ('man_out_taquicardia',               make_pattern(taquicardia_keywords)),
    ('man_out_bradicardia',               make_pattern(bradicardia_keywords)),
    ('man_out_extremidades_frias',        make_pattern(extremidades_keywords)),
    ('man_out_palidez',                   make_pattern(palidez_keywords)),
    ('man_out_sudorese',                  make_pattern(sudorese_keywords)),
    ('man_out_urticaria',                 make_pattern(urticaria_keywords)),
    ('man_out_broncoespasmo',             make_pattern(broncoespasmo_kw)),
    ('man_out_dispneia',                  make_pattern(dispneia_keywords)),
    ('man_out_angioedema',                make_pattern(angioedema_keywords)),
    ('man_out_tremor',                    make_pattern(tremor_keywords)),
    ('man_out_fotofobia',                 make_pattern(fotofobia_keywords)),
    ('man_out_visao_turva',               make_pattern(visao_turva_keywords)),
    ('man_out_guillain_barre',            make_pattern(guillain_barre_kw)),
    ('man_out_encefalite',                make_pattern(encefalite_keywords)),
    ('man_out_epilepsia',                 make_pattern(epilepsia_keywords)),
    ('man_out_paralisia',                 make_pattern(paralisia_keywords)),
    ('man_out_purpura_trombocitopenica',  make_pattern(purpura_keywords)),
]

for coluna, pattern in colunas_categorias_man_out:
    dados[coluna] = dados['man_out'].apply(
        lambda x, p=pattern: 1 if pd.notna(x) and
        re.search(p, normalizar(str(x)), re.IGNORECASE) else 0
    )

# ── checagem rápida ──────────────────────────────────────────────────────────
print(dados[[col for col, _ in colunas_categorias_man_out]].sum().sort_values(ascending=False))

man_out_emese                       76
man_out_nausea                      67
man_out_dor_abdominal               32
man_out_diarreia                    20
man_out_sudorese                     1
man_out_cefaleia                     1
man_out_exantema                     0
man_out_artralgia                    0
man_out_dor                          0
man_out_exantema_local               0
man_out_dor_no_corpo                 0
man_out_enduracao                    0
man_out_calor                        0
man_out_eritema                      0
man_out_edema                        0
man_out_prurido                      0
man_out_linfonodomegalia             0
man_out_lesao                        0
man_out_abscesso_quente              0
man_out_tontura                      0
man_out_sincope                      0
man_out_parestesia                   0
man_out_convulsao                    0
man_out_confusao_mental              0
man_out_fraqueza                     0
man_out_hipotensao       

/tmp/ipykernel_36993/2210757156.py:48: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  dados[coluna] = dados['man_out'].apply(
/tmp/ipykernel_36993/2210757156.py:48: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  dados[coluna] = dados['man_out'].apply(
/tmp/ipykernel_36993/2210757156.py:48: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame

**21. tp_atd**

Transformação em **tp_atd_ajustada**, que é junção das infos "Hospitalar, Observacao e Ambulatorio"

In [75]:
dados['tp_atd_ajustada'] = dados['tp_atd'].astype(str).str.extract(r'^([^-]+)')[0].str.strip().str.title()

print(dados['tp_atd_ajustada'].value_counts())

tp_atd_ajustada
Observacao     846
Ambulatorio    214
Hospitalar      23
Name: count, dtype: int64


/tmp/ipykernel_36993/3704858139.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  dados['tp_atd_ajustada'] = dados['tp_atd'].astype(str).str.extract(r'^([^-]+)')[0].str.strip().str.title()


**22. diag**

Transformação em **diag_ajustada**, que é junção das infos de sintomas


In [76]:
diag = dados['diag'].astype(str).str.upper()

conditions = [
    diag.str.strip() == '',
    diag.str.contains('DOR|EDEMA|ERITEMA|CELULITE|RUBOR|INCHAÇO'),
    diag.str.contains('FEBRE|NÁUSEA|VÔMIT|ÊMESE|DIARREIA'),
    diag.str.contains('SÍNCOPE|CONVULSÃO|TONTURA|ATAXIA|PARESIA|HIPOTONIA'),
    diag.str.contains('EXANTEMA|URTICÁRIA|PRURIDO|ALERGIA|ANAFILAXIA'),
    diag.str.contains('CEFALÉIA|FADIGA|SONOLÊNCIA|PALIDEZ'),
]
choices = ['sem diagnóstico', 'reação local', 'sintomas gastrointestinais',
           'sintomas neurológicos', 'reação alérgica', 'sintomas gerais']

dados['diag_ajustada'] = np.select(conditions, choices, default='outros')
print(dados['diag_ajustada'].value_counts())

diag_ajustada
sem diagnóstico               588
reação local                  165
outros                        117
sintomas neurológicos          70
sintomas gastrointestinais     66
reação alérgica                47
sintomas gerais                30
Name: count, dtype: int64


/tmp/ipykernel_36993/1965561989.py:14: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  dados['diag_ajustada'] = np.select(conditions, choices, default='outros')


In [77]:
print(dados['diag'].value_counts())

diag
                                                                                                                                                                                                                      588
SÍNCOPE                                                                                                                                                                                                                26
REAÇÃO NO LOCAL DE APLICAÇÃO                                                                                                                                                                                           15
EXANTEMA                                                                                                                                                                                                               14
CONVULSÃO                                                                                                                  

**23. cls_eien**

In [ ]:
import re

def ajustar_cls_ei(valor):
    if pd.isna(valor) or str(valor).strip() == "":
        return "Não se aplica"
    match = re.match(r"(A\.3\.[0-9]+)", str(valor))
    if match:
        return match.group(1)
    return str(valor)

dados["cls_eien"] = dados["cls_eien"].apply(ajustar_cls_ei)

/tmp/ipykernel_36993/4121729317.py:11: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  dados["cls_eien_ajustada"] = dados["cls_eien"].apply(ajustar_cls_ei)


cls_eien_ajustada
Não se aplica    696
A.3.7            249
A.3.4             59
A.3.5             25
A.3.9             19
A.3.8             12
A.3.2              9
A.3.1              7
A.3.6              5
A.3.3              2
Name: count, dtype: int64

**24. cls_fei**

In [119]:
dados['cls_fei'].value_counts()

cls_fei
                                  1066
dúvida sobre administração vac       3
DOSE A MAIS                          2
adm. 2 doses da mesma vacina         2
não é erro                           2
Dose adicional                       1
VACINA SEGREGADA?                    1
Celulite                             1
paciente inquieto                    1
Não é Erro de Imunização             1
Dose Adicional                       1
Vacina duplicada                     1
Não é considerado EI                 1
Name: count, dtype: int64

**25. cond_ei**

In [120]:
dados['cond_ei'].value_counts()

cond_ei
                                                                         693
Dose considerada válida                                                  225
Repetir a dose o mais rápido possível considerando o intervalo mínimo     81
Outros                                                                    62
Repetir a dose com aprazamento de reforço                                 11
Repetir a dose única o mais rápido possível                               10
Acompanhamento com dosagem de anticorpos                                   1
Name: count, dtype: int64

**24. diag_en**

In [123]:
import re

def ajustar_diag_en(valor):
    if pd.isna(valor) or str(valor).strip() == "":
        return "Não informado"
    
    partes = [p.strip() for p in str(valor).split("|")]
    
    # Filtra apenas os itens com código (67)
    itens_67 = [
        re.sub(r"^\(67\)\s*", "", p)          # remove o prefixo "(67) "
        for p in partes
        if re.match(r"^\(67\)", p)
    ]
    
    # Se não houver nenhum (67), pega todos os itens sem o código
    if not itens_67:
        itens_67 = [re.sub(r"^\(\d+\)\s*", "", p) for p in partes]
    
    # Remove duplicatas mantendo a ordem
    vistos = set()
    resultado = []
    for item in itens_67:
        if item not in vistos:
            vistos.add(item)
            resultado.append(item)
    
    return " | ".join(resultado)


dados["diag_en_ajustada"] = dados["diag_en"].apply(ajustar_diag_en)

# Garante que sobras de espaço ou variantes vazias também viram "Não informado"
dados["diag_en_ajustada"] = dados["diag_en_ajustada"].str.strip().replace("()", "Não informado")

print(dados["diag_en_ajustada"].value_counts())
print(dados['diag_en_ajustada'].unique().shape[0])

diag_en_ajustada
Não informado                                                                                                                                                                       666
SÍNCOPE                                                                                                                                                                              28
REAÇÃO NO LOCAL DE APLICAÇÃO                                                                                                                                                         18
EXANTEMA                                                                                                                                                                             13
HIPOTENSÃO                                                                                                                                                                            9
FEBRE                                                          

In [7]:
dados['diag_en'].value_counts()

diag_en
()                                                                                                                                                                                                       389
                                                                                                                                                                                                         277
(67) SÍNCOPE                                                                                                                                                                                              16
(67) EXANTEMA                                                                                                                                                                                              9
(67) HIPOTENSÃO                                                                                                                                                             

**25. cond_en**

In [125]:
def ajustar_cond_en(valor):
    if pd.isna(valor) or str(valor).strip() in ("", "()"):
        return "Não informado"
    
    partes = [p.strip() for p in str(valor).split("|")]
    
    # Filtra apenas os itens com código (67)
    itens_67 = [
        re.sub(r"^\(67\)\s*", "", p)
        for p in partes
        if re.match(r"^\(67\)", p)
    ]
    
    # Se não houver nenhum (67), retorna "Não informado" (ex: só código 25, 41, etc.)
    if not itens_67:
        return "Não informado"
    
    # Remove duplicatas mantendo a ordem
    vistos = set()
    resultado = []
    for item in itens_67:
        if item not in vistos:
            vistos.add(item)
            resultado.append(item)
    
    return " | ".join(resultado)


dados["cond_en"] = dados["cond_en"].apply(ajustar_cond_en)

print(dados["cond_en"].value_counts())

print(dados['cond_en'].unique().shape[0])

cond_en
Não informado                                                                    689
1 - Esquema mantido                                                              305
2 - Esquema mantido com precaução (ambiente hospitalar)                           61
5 - Esquema encerrado                                                             12
8 - Outros                                                                         8
4 - Contraindicação sem substituição de esquema                                    6
2 - Esquema mantido com precaução (ambiente hospitalar) | 1 - Esquema mantido      1
7 - Ignorado                                                                       1
Name: count, dtype: int64
8


**26. cls_en**

In [126]:
def ajustar_cls_en(valor):
    if pd.isna(valor) or str(valor).strip() in ("", "()"):
        return "Não informado"
    
    partes = [p.strip() for p in str(valor).split("|")]
    
    # Filtra itens com (67) e extrai apenas o código (ex: "A.1", "B.1", "D")
    itens_67 = []
    for p in partes:
        if re.match(r"^\(67\)", p):
            # Remove o prefixo "(67) " e pega só o código antes do " - "
            sem_prefixo = re.sub(r"^\(67\)\s*", "", p)
            codigo = sem_prefixo.split(" - ")[0].strip()
            itens_67.append(codigo)
    
    # Se não houver nenhum (67), retorna "Não informado"
    if not itens_67:
        return "Não informado"
    
    # Remove duplicatas mantendo a ordem
    vistos = set()
    resultado = []
    for item in itens_67:
        if item not in vistos:
            vistos.add(item)
            resultado.append(item)
    
    return " | ".join(resultado)


dados["cls_en"] = dados["cls_en"].apply(ajustar_cls_en)

print(dados["cls_en"].value_counts())
print(dados['cls_en'].unique().shape[0])

cls_en
Não informado      689
A.1                225
A.4                 63
C.1                 29
B.1                 25
C.2                 20
D                    7
A.1 | A.4            5
B.2                  5
A.1 | C.1            2
A.3                  2
A.1 | B.1            2
A.1 | B.1 | A.4      1
A.3 | B.1 | A.1      1
A.1 | C.2            1
A.4 | A.1            1
C.2 | B.1            1
C.1 | C.2 | A.1      1
A.4 | B.1            1
C.1 | A.1            1
C.2 | A.1            1
Name: count, dtype: int64
21


**27. cls_fed**

In [127]:
dados['cls_fed'].value_counts()

cls_fed
                    1076
Grave (EAG)            5
Não Grave (EANG)       2
Name: count, dtype: int64

In [ ]:
dados = dados.drop(columns=['cls_fed'])

**28. diad_enfed**

In [128]:
dados['diad_enfed'].value_counts()

diad_enfed
                                                                                         1076
(14) MENINGITE VIRAL                                                                        2
(67) ENCEFALITE | (41) ENCEFALITE                                                           1
(67) POLIARTRALGIA                                                                          1
(14) DERMATITE                                                                              1
(14) CEFALÉIA | (67) CEFALÉIA | (67) FEBRE | (14) FEBRE | (67) VÔMITOS | (14) VÔMITOS       1
(67) EPILEPSIA                                                                              1
Name: count, dtype: int64

In [ ]:
dados = dados.drop(columns=['diad_enfed'])

**29. cond_enfed**

In [129]:
dados['cond_enfed'].value_counts()

cond_enfed
                                                                                                                                                                                          1076
(14) 5 - Esquema encerrado -                                                                                                                                                                 2
(67) 1 - Esquema mantido -                                                                                                                                                                   2
(67) 5 - Esquema encerrado -  | (41) 5 - Esquema encerrado -                                                                                                                                 1
(14) 5 - Esquema encerrado -  | (67) 1 - Esquema mantido -  | (67) 1 - Esquema mantido -  | (14) 5 - Esquema encerrado -  | (67) 1 - Esquema mantido -  | (14) 5 - Esquema encerrado -       1
(14) 4 - Contraindicação sem subst

In [ ]:
dados = dados.drop(columns=['cond_enfed'])

**30. cls_enfed**

In [130]:
dados['cls_enfed'].value_counts()

cls_enfed
                                                                                                                                                                                                                                                                                                                                                 1076
(67) C.1 - Condições subjacentes ou emergentes | (41) C.1 - Condições subjacentes ou emergentes                                                                                                                                                                                                                                                     1
(14) B.1 - Relação temporal consistente, mas sem evidências na literatura para se estabelecer relação causal                                                                                                                                                                                                      

In [ ]:
dados = dados.drop(columns=['cls_enfed'])

**31. diag_compl**

In [33]:
def ajustar_diag_compl(valor):
    if pd.isna(valor) or str(valor).strip() in ("", "()", "()"):
        return "Não informado"
    
    valor = str(valor).strip()
    
    # Encontra onde começa a parte codificada "(XX) ..."
    primeiro_codigo = re.search(r'\(\d+\)', valor)
    
    if primeiro_codigo:
        parte_texto  = valor[:primeiro_codigo.start()]   # antes do primeiro código
        parte_codigo = valor[primeiro_codigo.start():]   # a partir do primeiro código
    else:
        parte_texto  = valor
        parte_codigo = ""
    
    # Extrai sintomas do texto livre (split por " | ")
    sintomas_texto = [
        s.strip() for s in parte_texto.split("|")
        if s.strip()
    ]
    
    # Extrai sintomas do código (67) na parte codificada
    sintomas_67 = [
        re.sub(r"^\(67\)\s*", "", p.strip())
        for p in parte_codigo.split("|")
        if re.match(r"^\(67\)", p.strip())
    ]
    
    # União: texto livre + (67) não duplicados, mantendo ordem
    vistos = set()
    resultado = []
    for item in sintomas_texto + sintomas_67:
        if item and item not in vistos:
            vistos.add(item)
            resultado.append(item)
    
    # Se não sobrou nada (ex: só havia código não-67), retorna "Não informado"
    return " | ".join(resultado) if resultado else "Não informado"


dados["diag_compl_ajustada"] = dados["diag_compl"].apply(ajustar_diag_compl)

print(dados["diag_compl_ajustada"].value_counts().head(20))
print(dados["diag_compl_ajustada"].unique().shape[0])

diag_compl_ajustada
Não informado                     588
SÍNCOPE                            26
REAÇÃO NO LOCAL DE APLICAÇÃO       14
EXANTEMA                           13
CONVULSÃO                           9
HIPOTENSÃO                          9
DOR NO LOCAL DA INJEÇÃO             8
FEBRE                               7
DESMAIO APÓS FORTE EMOÇÃO           7
URTICÁRIA                           6
SÍNCOPE OU REAÇÃO VASOVAGAL         6
EDEMA NO LOCAL DE APLICAÇÃO         6
ALERGIA                             6
VÔMITOS                             6
COMPLICAÇÃO DEVIDO A VACINAÇÃO      6
CEFALÉIA | VÔMITOS                  6
REAÇÃO ALÉRGICA                     5
PALIDEZ                             4
PRURIDO                             4
EXANTEMA | PRURIDO                  4
Name: count, dtype: int64
325


**32. cls_compl**

In [6]:
LABELS_TEXTO = [
    "Erro de Imunização (EI) - Com evento adverso",
    "Erro de Imunização (EI)",
    "Grave (EAG)",
    "Não Grave (EANG)",
    "Inclassificável",
    "Erro de Imunização",
]

def ajustar_cls_compl(valor):
    if pd.isna(valor) or str(valor).strip() == "":
        return "Não informado"

    valor_str = str(valor).strip()

    # CAMADA 1: Extrai códigos (67) → retorna só o código (A.1, B.1, C.2, D...)
    codigos_67 = re.findall(r'\(67\)\s*([A-D](?:\.\d+)?)\s*-', valor_str)
    if codigos_67:
        vistos = set()
        resultado = []
        for code in codigos_67:
            if code not in vistos:
                vistos.add(code)
                resultado.append(code)
        return " | ".join(resultado)

    # CAMADA 2: Tem códigos reais não-67 (ex: (14), (41)...) mas nenhum (67)
    # → sem informação útil sobre HPV
    if re.search(r'\(\d+\)', valor_str):
        return "Não informado"

    # CAMADA 3: Sem nenhum código → busca labels de texto conhecidos
    for label in LABELS_TEXTO:
        if label in valor_str:
            return label

    return "Não informado"


dados["cls_compl_ajustada"] = dados["cls_compl"].apply(ajustar_cls_compl)

print(dados["cls_compl_ajustada"].value_counts())
print(dados["cls_compl_ajustada"].unique().shape[0])

cls_compl_ajustada
Erro de Imunização (EI)                         496
A.1                                             225
Não Grave (EANG)                                154
A.4                                              63
C.1                                              29
B.1                                              24
Não informado                                    24
C.2                                              20
Grave (EAG)                                      12
D                                                 7
A.1 | A.4                                         5
B.2                                               5
Erro de Imunização (EI) - Com evento adverso      3
A.1 | C.1                                         2
A.3                                               2
A.1 | B.1                                         2
A.1 | B.1 | A.4                                   1
A.3 | B.1 | A.1                                   1
A.1 | C.2                                    

In [7]:
substituir = [
    'Erro de Imunização (EI)',
    'Não Grave (EANG)',
    'Grave (EAG)',
    'Erro de Imunização (EI) - Com evento adverso'
]

dados['cls_compl_ajustada'] = dados['cls_compl_ajustada'].apply(
    lambda x: 'Não informado' if x in substituir else x
)

# ── checagem ─────────────────────────────────────────────────────────────────
print(dados['cls_compl_ajustada'].value_counts())

cls_compl_ajustada
Não informado      689
A.1                225
A.4                 63
C.1                 29
B.1                 24
C.2                 20
D                    7
A.1 | A.4            5
B.2                  5
A.1 | C.1            2
A.3                  2
A.1 | B.1            2
A.1 | B.1 | A.4      1
A.3 | B.1 | A.1      1
A.1 | C.2            1
A.4 | A.1            1
C.1 | B.1            1
C.2 | B.1            1
C.1 | C.2 | A.1      1
A.4 | B.1            1
C.1 | A.1            1
C.2 | A.1            1
Name: count, dtype: int64


In [11]:
def classificar_cls_compl(valor):
    if pd.isna(valor) or valor == 'Não informado':
        return 'Não informado'

    # separa as categorias presentes no valor
    cats = set(c.strip() for c in valor.split('|'))

    # ── A.3 prevalece sempre ─────────────────────────────────────────────────
    if 'A.3' in cats:
        return 'A.3'

    # ── A.1 e A.4 juntos → revisão manual ───────────────────────────────────
    if 'A.1' in cats and 'A.4' in cats:
        return 'Revisar manualmente'

    # ── A prevalece sobre B e C ──────────────────────────────────────────────
    if 'A.1' in cats:
        return 'A.1'
    if 'A.4' in cats:
        return 'A.4'

    # ── C prevalece sobre B ──────────────────────────────────────────────────
    # se houver múltiplos Cs, mantém ambos para revisão
    cs_presentes = [c for c in cats if c.startswith('C')]
    if len(cs_presentes) > 1:
        return 'Revisar manualmente'
    if len(cs_presentes) == 1:
        return cs_presentes[0]

    # ── somente B ────────────────────────────────────────────────────────────
    bs_presentes = [c for c in cats if c.startswith('B')]
    if len(bs_presentes) == 1:
        return bs_presentes[0]

    # ── D ou outros valores simples ──────────────────────────────────────────
    if len(cats) == 1:
        return list(cats)[0]

    return 'A.1 e A.4'


dados['cls_compl_ajustada'] = dados['cls_compl_ajustada'].apply(classificar_cls_compl)

# ── checagem ─────────────────────────────────────────────────────────────────
print(dados['cls_compl_ajustada'].value_counts())

cls_compl_ajustada
Não informado          689
A.1                    233
A.4                     64
C.1                     30
B.1                     24
C.2                     21
D                        7
Revisar manualmente      7
B.2                      5
A.3                      3
Name: count, dtype: int64


**33. evol_num**

In [133]:
print(dados["evol_num"].value_counts())

evol_num
Cura sem sequelas      372
Em acompanhamento      100
Cura com sequelas       11
Não é EAVP               7
Perda de seguimento      5
Name: count, dtype: int64


## Ajuste Final para cada manifestação

In [81]:
categorias_finais = [
    'dor_abdominal', 'dor_no_corpo', 'artralgia', 'cefaleia', 'dor',
    'exantema_local', 'exantema', 'edema', 'eritema', 'calor', 'enduracao',
    'abscesso_quente', 'lesao', 'linfonodomegalia', 'prurido', 'febre',
    'nausea', 'emese', 'diarreia', 'tontura', 'sincope', 'parestesia',
    'convulsao', 'confusao_mental', 'fraqueza', 'hipotensao', 'taquicardia',
    'bradicardia', 'extremidades_frias', 'palidez', 'sudorese', 'urticaria',
    'broncoespasmo', 'dispneia', 'angioedema', 'tremor', 'fotofobia',
    'visao_turva', 'guillain_barre', 'encefalite', 'epilepsia', 'paralisia',
    'purpura_trombocitopenica'
]

prefixos = ['esavi_', 'man_loc_', 'man_sis_', 'man_out_']

for cat in categorias_finais:
    colunas = [f'{prefixo}{cat}' for prefixo in prefixos]
    dados[cat] = dados[colunas].max(axis=1)

# ── checagem rápida ──────────────────────────────────────────────────────────
print(dados[categorias_finais].sum().sort_values(ascending=False))

dor                         186
eritema                     149
edema                       132
emese                       108
febre                       105
palidez                     100
cefaleia                     94
sincope                      94
calor                        89
nausea                       88
prurido                      84
exantema                     63
urticaria                    54
tontura                      48
hipotensao                   44
fraqueza                     43
dor_abdominal                38
abscesso_quente              34
convulsao                    30
sudorese                     30
diarreia                     27
lesao                        24
extremidades_frias           21
dispneia                     17
parestesia                   17
dor_no_corpo                 15
paralisia                    13
visao_turva                  11
tremor                       11
angioedema                   10
enduracao                     9
taquicar

/tmp/ipykernel_1847/819742451.py:17: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  dados[cat] = dados[colunas].max(axis=1)
/tmp/ipykernel_1847/819742451.py:17: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  dados[cat] = dados[colunas].max(axis=1)
/tmp/ipykernel_1847/819742451.py:17: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use 

In [82]:
# ── manifestações locais ─────────────────────────────────────────────────────

locais = [
    'dor', 'edema', 'eritema', 'calor', 'enduracao', 'abscesso_quente',
    'lesao', 'linfonodomegalia', 'prurido', 'exantema_local'
]

dados['manifestacoes_locais'] = dados[locais].max(axis=1)

# ── manifestações sistêmicas ─────────────────────────────────────────────────

sistemicas = [
    'febre', 'cefaleia', 'dor_abdominal', 'dor_no_corpo', 'artralgia',
    'nausea', 'emese', 'diarreia', 'tontura', 'sincope', 'parestesia',
    'convulsao', 'confusao_mental', 'fraqueza', 'hipotensao', 'taquicardia',
    'bradicardia', 'extremidades_frias', 'palidez', 'sudorese', 'urticaria',
    'exantema', 'broncoespasmo', 'dispneia', 'angioedema', 'tremor',
    'fotofobia', 'visao_turva', 'guillain_barre', 'encefalite', 'epilepsia',
    'paralisia', 'purpura_trombocitopenica'
]

dados['manifestacoes_sistemicas'] = dados[sistemicas].max(axis=1)

# ── checagem ─────────────────────────────────────────────────────────────────
print("Manifestações locais:   ", dados['manifestacoes_locais'].sum())
print("Manifestações sistêmicas:", dados['manifestacoes_sistemicas'].sum())
print("\nAmbas simultaneamente:  ", (dados['manifestacoes_locais'] & dados['manifestacoes_sistemicas']).sum())

Manifestações locais:    296
Manifestações sistêmicas: 420

Ambas simultaneamente:   175


/tmp/ipykernel_1847/3779022073.py:8: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  dados['manifestacoes_locais'] = dados[locais].max(axis=1)
/tmp/ipykernel_1847/3779022073.py:22: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  dados['manifestacoes_sistemicas'] = dados[sistemicas].max(axis=1)
